In [ ]:
from llama_index.llms.langchain import LangChainLLM
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1:8b",
    base_url="",
    temperature=0.7
)

llama_llm = LangChainLLM(llm=llm)

In [ ]:
from llama_index.core import Document

docs = [
    Document(text="Paris is the capital of France."),
    Document(text="France is a country in Europe."),
]

In [ ]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(
    docs,
    llm=llama_llm
)

In [ ]:
query_engine = index.as_query_engine()

response = query_engine.query("What is the capital of France?")
print(response)

In [ ]:
from llama_index.core.agent import ReActAgent

agent = ReActAgent.from_tools(
    tools=[],
    llm=llama_llm
)

response = agent.chat("Explain the capital of France with context")
print(response)

In [9]:
from langchain_ollama import ChatOllama

OLLAMA_BASE_URL = "http://192.168.1.120:11434"
MODEL_NAME = "llama3.1:8b"

query = "What is the capital of France?"

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=0.7
)

response = llm.invoke(query)
print(response.content)

The capital of France is Paris.


In [10]:
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
from crewai import Agent


researcher = Agent(
    role="Researcher",
    goal="Find accurate factual information",
    backstory="You are a meticulous researcher who verifies facts.",
    llm=llm,
    verbose=True
)

writer = Agent(
    role="Writer",
    goal="Explain information clearly and concisely",
    backstory="You are a technical writer who simplifies complex ideas.",
    llm=llm,
    verbose=True
)


In [12]:
from crewai import Task

research_task = Task(
    description="Find the capital of France.",
    expected_output="The capital of France with a short explanation.",
    agent=researcher
)

writing_task = Task(
    description="Explain the answer in one simple sentence.",
    expected_output="A single clear sentence.",
    agent=writer
)


In [13]:
from crewai import Crew

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    process="sequential",  # one task after another
    verbose=True
)

In [ ]:
result = crew.kickoff()
print(result)

In [15]:
manager = Agent(
    role="Manager",
    goal="Coordinate agents and ensure correct results",
    backstory="You manage a team and delegate tasks intelligently.",
    llm=llm,
    verbose=True
)

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    manager_llm=llm,
    process="hierarchical",
    verbose=True
)

In [16]:
from crewai.tools import tool

@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

math_agent = Agent(
    role="Math Expert",
    goal="Solve numerical problems",
    backstory="You are precise and logical.",
    llm=llm,
    tools=[add_numbers],
    verbose=True
)

In [17]:
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    process="sequential",
    memory=True,   # 🔥 enable memory
    verbose=True
)

def task_callback(task_output):
    print("\n--- TASK COMPLETED ---")
    print(task_output)

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    process="sequential",
    callbacks=[task_callback],
    verbose=True
)

In [ ]:
from crewai import Agent, Task, Crew
from crewai.tools import tool
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1:8b",
    base_url="http://192.168.1.120:11434",
    temperature=0.7
)

@tool
def lookup_country_capital(country: str) -> str:
    """Return the capital of a country"""
    data = {"France": "Paris"}
    return data.get(country, "Unknown")

researcher = Agent(
    role="Researcher",
    goal="Find correct capitals",
    backstory="Fact-driven researcher",
    llm=llm,
    tools=[lookup_country_capital],
    verbose=True
)

writer = Agent(
    role="Writer",
    goal="Explain clearly",
    backstory="Simple and precise communicator",
    llm=llm,
    verbose=True
)

task1 = Task(
    description="Find the capital of France.",
    expected_output="Capital name.",
    agent=researcher
)

task2 = Task(
    description="Explain the capital in one sentence.",
    expected_output="Clear explanation.",
    agent=writer
)

crew = Crew(
    agents=[researcher, writer],
    tasks=[task1, task2],
    process="sequential",
    memory=True,
    verbose=True
)

result = crew.kickoff()
print(result)
